# Stage 0 — RSA Verification (Representational Similarity Analysis)

This notebook answers the gate question from
`notes/research_question/03_pivot2_group_consistency.md` §12.4 /
`notes/simple_notes.md` §3-4:

> If two demographics (or two questions) differ but their real survey answers
> are similar — are their representations/embeddings inside the LLM's "brain"
> also close together?

- If **yes** (RSA passes) → we may go on to design `L_group`/`L_question` using
  weights taken straight from the LLM representations (Stage 1).
- If **no** (RSA fails) → the raw LLM representations cannot be trusted for
  this, and we must fall back to statistically learned weights.

**Model used:** `mistralai/Mistral-7B-v0.1` — chosen because it is the only
model used **in exactly the same way** in both of our reference papers
(SubPOP and llm-opinions / "What Do Large Language Models Know About
Opinions?"), so we can directly reuse their layer-choice findings without
re-deriving them from scratch.

**How to use:** first check the Kaggle settings in the next markdown cell, then
`Run All`. Nothing needs manual editing unless the data path turns out to be
different from the automatic guess below.

## Before running: Kaggle settings you must check

1. **Accelerator**: three-dot menu at the top right → *Notebook options* → pick
   **GPU T4 x2** or **GPU P100** (not CPU / TPU).
2. **Internet**: in the same panel, turn on **Internet: On** (needed to
   download the Mistral-7B-v0.1 model from HuggingFace, ~14GB, once at the start).
3. **Upload data** — this notebook needs the `opinionqa.csv` file from the repo
   (original path: `datasets/subpop/data/opinionqa/processed/opinionqa.csv`):
   - Right panel → **Add Data** → **Upload** → pick the `opinionqa.csv` file →
     give the dataset a name → **Create**.
   - Once *attached*, it shows up automatically at
     `/kaggle/input/<dataset-name>/opinionqa.csv` and the config cell below
     will find it by itself (it searches the pattern `**/opinionqa.csv`).
   - If it turns out it is not found automatically, fill in the
     `OPINIONQA_CSV_PATH` variable manually in the config cell.

Estimated total runtime: **15–25 minutes** (mostly the one-off download +
model load at the start; extracting representations for a few hundred prompts
takes only a few minutes, and the rest is lightweight statistics).

In [ ]:
# Idempotent — safe to re-run.
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm
!pip install -q -U bitsandbytes  # only needed if USE_4BIT=True in the config cell

In [ ]:
import os
import ast
import glob
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    raise RuntimeError(
        "No GPU detected. Check the three-dot menu at the top right -> Notebook options -> "
        "Accelerator -> pick GPU T4 x2 or P100, then restart the session & Run All again."
    )

In [ ]:
# ── Model ────────────────────────────────────────────────────────
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False   # set True if you hit OOM (e.g. you only got a single T4 16GB)

# ── Data ─────────────────────────────────────────────────────────
# Automatically look for the opinionqa.csv file among the attached Kaggle Datasets.
_candidates = glob.glob("/kaggle/input/**/opinionqa.csv", recursive=True)
if _candidates:
    OPINIONQA_CSV_PATH = _candidates[0]
elif os.path.exists("opinionqa.csv"):
    OPINIONQA_CSV_PATH = "opinionqa.csv"
else:
    raise FileNotFoundError(
        "Could not find opinionqa.csv. Upload this file as a Kaggle Dataset first "
        "(Add Data -> Upload -> opinionqa.csv), attach it to the notebook, then re-run "
        "this cell. If the folder name differs from the guess, fill it in manually: "
        "OPINIONQA_CSV_PATH = '/kaggle/input/.../opinionqa.csv'"
    )
print("Using data from:", OPINIONQA_CSV_PATH)

# ── Analysis scope ───────────────────────────────────────────────
# None = use ALL available data (more accurate, and still fast).
# Put in a number (e.g. 100) if you only want a quick trial run / debugging.
N_QKEYS_FOR_GROUP_WD = None   # number of questions used to compute the real distance between GROUPS
N_QUESTIONS_SAMPLE   = None   # number of questions that are themselves members of the QUESTION axis

RANDOM_SEED = 42
N_PERMUTATIONS = 2000  # for the significance test (permutation / Mantel test) at the best layer

# ── Output ───────────────────────────────────────────────────────
# MUST be under /kaggle/working -- that is the only writable folder.
# /kaggle/input (where the uploaded dataset lives) is always read-only.
OUT_DIR = "/kaggle/working/stage0_rsa"
assert not OUT_DIR.startswith("/kaggle/input"), (
    "OUT_DIR must not be under /kaggle/input -- that is read-only (where the uploaded dataset lives). "
    "The only place files may be written is under /kaggle/working/."
)
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
df = pd.read_csv(OPINIONQA_CSV_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"]   = df["ordinal"].apply(_parse_list)
df["options"]   = df["options"].apply(_parse_list)

# Drop the "Overall" rows — that is the aggregate over everyone, not a demographic group.
df = df[df["attribute"] != "Overall"].reset_index(drop=True)
df["group_label"] = df["attribute"] + ": " + df["group"].astype(str)

print(f"Total rows: {len(df)}")
print(f"Number of unique demographic groups: {df['group_label'].nunique()}")
print(f"Number of unique questions (qkey): {df['qkey'].nunique()}")

In [ ]:
def ordinal_weighted_mean(responses, ordinal):
    """Average score of one group on that question's scale
    (a larger score = leaning towards the options with larger codes)."""
    r = np.asarray(responses, dtype=float)
    o = np.asarray(ordinal, dtype=float)
    if r.sum() == 0:
        return np.nan
    return float(np.sum(r * o) / r.sum())

df["scalar_score"] = df.apply(lambda row: ordinal_weighted_mean(row["responses"], row["ordinal"]), axis=1)

GROUP_LABELS = sorted(df["group_label"].unique().tolist())
print(f"{len(GROUP_LABELS)} demographic groups:")
print(GROUP_LABELS)

# table (group x question) holding those scalar scores -- used for the QUESTION axis later
scalar_pivot = df.pivot_table(index="group_label", columns="qkey", values="scalar_score")
print("\nShape of the score table (group x question):", scalar_pivot.shape)

## 1. Compute the "real distance" (from the human survey data)

- **Group axis**: the average Wasserstein Distance (exactly the same distance
  metric SubPOP uses) between the answer distributions of two groups,
  averaged over all the questions both of them answered.
- **Question axis**: two questions count as "logically similar" if their
  answer patterns **across groups** are similar (e.g. if the same segments
  that tend to agree on question A also tend to agree on question B, even
  though the topics differ) — measured via the correlation between the columns
  of the score table above, then `distance = 1 - correlation`.

In [ ]:
resp_lookup = {
    (gl, qk): (resp, ordv)
    for gl, qk, resp, ordv in zip(df["group_label"], df["qkey"], df["responses"], df["ordinal"])
}

rng = np.random.default_rng(RANDOM_SEED)
all_qkeys = df["qkey"].unique().tolist()

if N_QKEYS_FOR_GROUP_WD is None:
    sample_qkeys_for_wd = all_qkeys
else:
    sample_qkeys_for_wd = list(rng.choice(all_qkeys, size=min(N_QKEYS_FOR_GROUP_WD, len(all_qkeys)), replace=False))

n_g = len(GROUP_LABELS)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Computing real distances between groups"):
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gi, gj = GROUP_LABELS[i], GROUP_LABELS[j]
        wds = []
        for qk in sample_qkeys_for_wd:
            ri = resp_lookup.get((gi, qk))
            rj = resp_lookup.get((gj, qk))
            if ri is None or rj is None:
                continue
            respA, ordA = ri
            respB, ordB = rj
            if len(ordA) != len(ordB):
                continue  # just in case the option scales differ (should rarely happen)
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds)) if wds else np.nan

print("\nExample: the 6 groups most similar to the first group according to the real answers:")
print(pd.Series(group_real_dist[0], index=GROUP_LABELS).sort_values().head(7))

In [ ]:
if N_QUESTIONS_SAMPLE is None:
    QUESTION_SAMPLE = all_qkeys
else:
    QUESTION_SAMPLE = list(rng.choice(all_qkeys, size=min(N_QUESTIONS_SAMPLE, len(all_qkeys)), replace=False))

question_corr = scalar_pivot[QUESTION_SAMPLE].corr(method="pearson", min_periods=10)
question_real_dist = (1.0 - question_corr.values)

qtext_lookup = df.drop_duplicates("qkey").set_index("qkey")[["question", "options"]].to_dict("index")

print(f"{len(QUESTION_SAMPLE)} questions used for the question axis.")
print("Shape of the real distance matrix between questions:", question_real_dist.shape)

## 2. Build a prompt for each group & each question

The group prompt is nothing but a plain demographic description (without any
survey question at all) — so that the representation we extract is purely
"the concept of that group", not mixed up with the context of one particular
question. The question prompt uses the standard survey format (question +
letter-labelled options + `Answer:`), consistent with how SubPOP/llm-opinions
format survey questions for an LLM.

In [ ]:
def build_group_prompt(group_label):
    attribute, group = group_label.split(": ", 1)
    return f"Survey respondent profile.\n{attribute}: {group}."

def build_question_prompt(qkey):
    info = qtext_lookup[qkey]
    q, opts = info["question"], info["options"]
    letters = [chr(ord("A") + i) for i in range(len(opts))]
    lines = [q] + [f"{letter}. {opt}" for letter, opt in zip(letters, opts)] + ["Answer:"]
    return "\n".join(lines)

group_prompts = {g: build_group_prompt(g) for g in GROUP_LABELS}
question_prompts = {q: build_question_prompt(q) for q in QUESTION_SAMPLE}

print("Example GROUP prompt:\n" + list(group_prompts.values())[0])
print("\n---\n")
print("Example QUESTION prompt:\n" + list(question_prompts.values())[0])

## 3. Load the model & extract the internal representations

We take the representation of the **last** token at **every layer**
(`output_hidden_states=True` from plain Transformers) — the same raw material
used by `llm_opinions/utils/activation_utils.py` (probing) in the llm-opinions
paper, only rewritten more compactly here because we do not need their
per-attention-head features or steering hooks, just the per-layer
representations.

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH} (the first download can take a few minutes, ~14GB)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = dict(torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Number of layers: {NUM_LAYERS} (+1 for the initial embedding before layer 1)")

In [ ]:
@torch.no_grad()
def get_hidden_states_all_layers(prompt: str) -> np.ndarray:
    """Representation of the LAST token at every layer.
    Output shape: (num_layers+1, hidden_size) -- index 0 = the initial embedding.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    hs = torch.stack(out.hidden_states, dim=0)   # (num_layers+1, batch=1, seq_len, hidden)
    return hs[:, 0, -1, :].float().cpu().numpy()  # (num_layers+1, hidden)

group_embeddings = {}
for g, prompt in tqdm(group_prompts.items(), desc="Extracting group representations"):
    group_embeddings[g] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

question_embeddings = {}
for q, prompt in tqdm(question_prompts.items(), desc="Extracting question representations"):
    question_embeddings[q] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array = np.stack([group_embeddings[g] for g in GROUP_LABELS])
question_emb_array = np.stack([question_embeddings[q] for q in QUESTION_SAMPLE])

np.savez(
    os.path.join(OUT_DIR, "embeddings_raw.npz"),
    group_emb=group_emb_array,
    question_emb=question_emb_array,
    group_labels=np.array(GROUP_LABELS, dtype=object),
    question_ids=np.array(QUESTION_SAMPLE, dtype=object),
)
print("Raw representations saved to", os.path.join(OUT_DIR, "embeddings_raw.npz"))

## 4. RSA: compare the LLM-representation distances vs the real distances

The core statistic: the **Spearman** correlation between two distance matrices
(this is the RSA definition from Kriegeskorte et al. 2008 — see
`notes/research_question/03_pivot2_group_consistency.md` §12.5). For the
p-value we use a **permutation test** (shuffle the matrix ordering many times
and see how often a result that random comes out >= our real result), not the
standard Spearman p-value — because the entries inside a single distance
matrix are interrelated (not independent samples), so the standard p-value
would be far too confident. This is the standard technique in RSA; it is
called the **Mantel test**.

In [ ]:
def upper_tri(mat):
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

def rho_only(emb_array, real_dist_matrix, layer_idx):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    real_flat, rep_flat = upper_tri(real_dist_matrix), upper_tri(rep_dist)
    mask = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    rho, _ = spearmanr(real_flat[mask], rep_flat[mask])
    return rho

def rsa_with_permutation_test(emb_array, real_dist_matrix, layer_idx, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    real_flat, rep_flat = upper_tri(real_dist_matrix), upper_tri(rep_dist)
    mask = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    real_flat, rep_flat = real_flat[mask], rep_flat[mask]
    rho, _ = spearmanr(real_flat, rep_flat)

    rng_p = np.random.default_rng(seed)
    n = real_dist_matrix.shape[0]
    perm_rhos = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng_p.permutation(n)
        permuted_flat = upper_tri(real_dist_matrix[np.ix_(perm, perm)])[mask]
        r, _ = spearmanr(permuted_flat, rep_flat)
        perm_rhos[k] = 0.0 if np.isnan(r) else r

    p_value = float(np.mean(np.abs(perm_rhos) >= abs(rho)))
    return float(rho), p_value

n_layers_total = group_emb_array.shape[1]
group_rhos    = [rho_only(group_emb_array, group_real_dist, L) for L in range(n_layers_total)]
question_rhos = [rho_only(question_emb_array, question_real_dist, L) for L in range(n_layers_total)]

results_df = pd.DataFrame({
    "layer": list(range(n_layers_total)),
    "rsa_rho_group_axis": group_rhos,
    "rsa_rho_question_axis": question_rhos,
})
print(results_df.to_string())

In [ ]:
best_group_layer    = int(np.nanargmax(np.abs(group_rhos)))
best_question_layer = int(np.nanargmax(np.abs(question_rhos)))

print(f"Best layer, GROUP axis   : {best_group_layer}  (rho={group_rhos[best_group_layer]:.3f})")
print(f"Best layer, QUESTION axis: {best_question_layer}  (rho={question_rhos[best_question_layer]:.3f})")
print(f"\nRunning the permutation test ({N_PERMUTATIONS}x shuffles) at the best layer of each axis...")

group_rho_best, group_p_best = rsa_with_permutation_test(group_emb_array, group_real_dist, best_group_layer)
question_rho_best, question_p_best = rsa_with_permutation_test(question_emb_array, question_real_dist, best_question_layer)

print(f"\n[GROUP AXIS]    layer {best_group_layer}: rho={group_rho_best:.3f}, p={group_p_best:.4f}")
print(f"[QUESTION AXIS] layer {best_question_layer}: rho={question_rho_best:.3f}, p={question_p_best:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df["layer"], results_df["rsa_rho_group_axis"], marker="o", label="Group axis (demographics)")
ax.plot(results_df["layer"], results_df["rsa_rho_question_axis"], marker="s", label="Question axis")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(15, color="red", linestyle="--", alpha=0.5, label="layer ~15 ('deep enough' threshold per llm-opinions)")
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho\n(LLM-representation distance vs real-answer distance)")
ax.set_title("Stage 0 -- RSA check: do the LLM representations match the similarity of the real survey answers?")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_rho_vs_layer.png"), dpi=150)
plt.show()

In [ ]:
def verdict(rho, p, label, loss_name):
    if p >= 0.05:
        return (f"[FAIL - not significant] {label}: rho={rho:.3f}, p={p:.4f} -- no evidence of a relationship "
                f"we could rely on (it may just be random chance).")
    if rho > 0.15:
        return (f"[PASS] {label}: rho={rho:.3f}, p={p:.4f} -- the LLM representations are trustworthy enough "
                f"to be used as proximity weights for {loss_name}.")
    if rho < -0.15:
        return (f"[FAIL - DIRECTION REVERSED] {label}: rho={rho:.3f}, p={p:.4f} -- there is a strong and "
                f"significant relationship, BUT ITS DIRECTION IS REVERSED from what we expected (representations grow "
                f"more different precisely for the groups/questions whose real answers grow more similar). Do NOT use it as "
                f"a proximity weight before investigating -- it could be a prompt-design artefact, or it could be a real finding.")
    return (f"[SIGNIFICANT BUT WEAK] {label}: rho={rho:.3f}, p={p:.4f} -- there is a faint signal, probably "
            f"only detected because the sample is large (not a practically strong relationship). "
            f"Consider combining it with statistical weights; do not rely on the raw representations alone.")

print(verdict(group_rho_best, group_p_best, "Group axis", "L_group"))
print(verdict(question_rho_best, question_p_best, "Question axis", "L_question"))

In [ ]:
results_df.to_csv(os.path.join(OUT_DIR, "rsa_rho_per_layer.csv"), index=False)

summary = pd.DataFrame([
    {"axis": "group", "best_layer": best_group_layer, "rho": group_rho_best, "p_value": group_p_best,
     "n_items": len(GROUP_LABELS)},
    {"axis": "question", "best_layer": best_question_layer, "rho": question_rho_best, "p_value": question_p_best,
     "n_items": len(QUESTION_SAMPLE)},
])
summary.to_csv(os.path.join(OUT_DIR, "rsa_summary.csv"), index=False)
print(summary.to_string(index=False))

print(f"\nAll outputs are in: {OUT_DIR}")
print("Download this folder from the Kaggle 'Output' tab before the session ends,")
print("then bring the numbers back to update research_question/03_pivot2_group_consistency.md paragraph 12.9 / TODO.md.")

## Done — how to read the results

- **rho (Spearman correlation)**: the larger & more positive, the better the
  LLM representation distances match the real answer distances. Range -1 to 1.
- **p-value**: from the *permutation test* (Mantel test), not the standard
  Spearman p-value — see the explanation above the RSA cell.
- The thresholds in the verdict cell (**p < 0.05 and rho > 0.15 → PASS**) are
  a reasonable starting point, not sacred numbers. If a result lands near
  those thresholds (e.g. rho=0.14, p=0.04), read the numbers manually instead
  of merely trusting the automatic label.

**Next steps:**
1. Download the `/kaggle/working/stage0_rsa/` folder (*Output* button in the right panel).
2. Bring `rsa_summary.csv` + `rsa_rho_vs_layer.png` back to the local repo.
3. Update `notes/research_question/03_pivot2_group_consistency.md` §12.9
   (the Stage 0 checklist) and `TODO.md` PRIORITY #1 with the results —
   pass/fail, at which layer, for both axes.
4. If it **passes** → move on to Stage 1 (designing `L_group`/`L_question`).
   If it **fails** → read `notes/simple_notes.md` section 4, "the RSA-is-wrong
   scenario", for the fallback plan.

## Stage 0b (diagnostic) — is the negative group-axis rho a prompt artefact?

Result of the first run: the group axis got a **significant negative rho**
(not merely weak/zero). Before concluding "the LLM representations
misunderstand demographic similarity", let us first check the cheaper
possibility: perhaps it is only an artefact of the `"{ATTRIBUTE}: {group}."`
prompt template — because groups within the **same attribute** (e.g. the 5
`POLIDEOLOGY` groups) are nearly identical on the surface as text (differing
by only 1-2 words), so their representations end up close automatically,
**even though** groups within one attribute are often the very ones whose
opinions differ most (e.g. Very Liberal vs Very Conservative).

The cell below rebuilds the group prompts using natural sentences (without
naming internal survey attribute codes like `AGE:`/`POLIDEOLOGY:`), then
re-extracts the representations **using the same already-loaded model** (no
reload needed), and recomputes the RSA for the group axis only. The real
distances (`group_real_dist`) do not change — only the representation side is
swapped out.

In [ ]:
GROUP_TEMPLATES_V2 = {
    "AGE": lambda g: f"This survey respondent is {g} years old.",
    "CITIZEN": lambda g: f"This survey respondent is {'a U.S. citizen' if g == 'Yes' else 'not a U.S. citizen'}.",
    "CREGION": lambda g: f"This survey respondent lives in the {g} region of the United States.",
    "EDUCATION": lambda g: f"This survey respondent's highest level of education is {g}.",
    "INCOME": lambda g: f"This survey respondent's household income is {g}.",
    "MARITAL": lambda g: f"This survey respondent's marital status is {g}.",
    "POLIDEOLOGY": lambda g: f"Politically, this survey respondent describes their views as {g.lower()}.",
    "POLPARTY": lambda g: f"This survey respondent's political party affiliation is {g}.",
    "RACE": lambda g: f"This survey respondent's race is {g}.",
    "RELIG": lambda g: f"This survey respondent's religion is {g}.",
    "RELIGATTEND": lambda g: f"This survey respondent attends religious services: {g.lower()}.",
    "SEX": lambda g: f"This survey respondent is {g.lower()}.",
}

def build_group_prompt_v2(group_label):
    attribute, group = group_label.split(": ", 1)
    return GROUP_TEMPLATES_V2[attribute](group)

group_prompts_v2 = {g: build_group_prompt_v2(g) for g in GROUP_LABELS}

print("Example comparison of prompt v1 (old) vs v2 (natural, without attribute names):\n")
for g in GROUP_LABELS[:4] + ["POLIDEOLOGY: Very liberal", "POLIDEOLOGY: Very conservative"]:
    print(f"v1: {group_prompts[g]!r}")
    print(f"v2: {group_prompts_v2[g]!r}")
    print()

In [ ]:
group_embeddings_v2 = {}
for g, prompt in tqdm(group_prompts_v2.items(), desc="Extracting group representations (v2, natural)"):
    group_embeddings_v2[g] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array_v2 = np.stack([group_embeddings_v2[g] for g in GROUP_LABELS])

np.savez(
    os.path.join(OUT_DIR, "embeddings_group_v2.npz"),
    group_emb_v2=group_emb_array_v2,
    group_labels=np.array(GROUP_LABELS, dtype=object),
)
print("v2 representations saved to", os.path.join(OUT_DIR, "embeddings_group_v2.npz"))

In [ ]:
group_rhos_v2 = [rho_only(group_emb_array_v2, group_real_dist, L) for L in range(n_layers_total)]

compare_df = pd.DataFrame({
    "layer": list(range(n_layers_total)),
    "rho_v1_attr_code_prefix": group_rhos,
    "rho_v2_natural_sentence": group_rhos_v2,
})
print(compare_df.to_string())

best_group_layer_v2 = int(np.nanargmax(np.abs(group_rhos_v2)))
group_rho_best_v2, group_p_best_v2 = rsa_with_permutation_test(group_emb_array_v2, group_real_dist, best_group_layer_v2)

print(f"\n[v1 - 'ATTR: value' format]  layer {best_group_layer}: rho={group_rho_best:.3f}, p={group_p_best:.4f}")
print(f"[v2 - natural sentence]      layer {best_group_layer_v2}: rho={group_rho_best_v2:.3f}, p={group_p_best_v2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(compare_df["layer"], compare_df["rho_v1_attr_code_prefix"], marker="o", label="v1 -- 'ATTR: value' format")
ax.plot(compare_df["layer"], compare_df["rho_v2_natural_sentence"], marker="o", label="v2 -- natural sentence")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho (group axis)")
ax.set_title("Stage 0b -- effect of prompt design on the group-axis rho")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_group_prompt_comparison.png"), dpi=150)
plt.show()

compare_df.to_csv(os.path.join(OUT_DIR, "rsa_group_prompt_comparison.csv"), index=False)

print(verdict(group_rho_best_v2, group_p_best_v2, "Group axis (prompt v2, natural)", "L_group"))
print()
if group_rho_best < -0.1 and group_rho_best_v2 > 0.05:
    print("=> The sign flips from negative (v1) to positive (v2) -- strong indication that v1 really was a prompt-template artefact, "
          "not the LLM misunderstanding demographic similarity. Carry on with v2-style prompts for Stage 1.")
elif group_rho_best_v2 < -0.1:
    print("=> Still negative & fairly strong in v2 as well -- the prompt-artefact hypothesis is NOT confirmed; this is likely "
          "a real finding: for this model the LLM representations genuinely do not capture demographic similarity correctly.")
else:
    print("=> The result changed but is not entirely clear (neither clearly strongly positive nor strongly negative) -- "
          "read the numbers manually; probably a combination of a prompt artefact AND representation limitations.")